# U-Net Image Segmentation Model Architecture Explaination.

### 1. ```DoubleConv``` class



In [36]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


### 2. ```UNet``` class


In [37]:
class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()


### Encoder (Contracting Path)

In [38]:
import torch
import torch.nn as nn

n_channels = 1      # use 3 if RGB images
n_classes  = 1      # number of output classes

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes):
        super().__init__()

        self.inc = DoubleConv(n_channels, 64)

        self.down1 = nn.MaxPool2d(2)
        self.conv1 = DoubleConv(64, 128)

        self.down2 = nn.MaxPool2d(2)
        self.conv2 = DoubleConv(128, 256)

        self.down3 = nn.MaxPool2d(2)
        self.conv3 = DoubleConv(256, 512)

        self.down4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)


### Decoder (Expansive Path)

In [39]:
import torch
import torch.nn as nn

n_channels = 1      # use 3 if RGB images
n_classes  = 1      # number of output classes

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes):
        super().__init__()
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv5 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv6 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv7 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, n_classes, 1)

### 3. ```forward()``` method



In [40]:
def forward(self, x):
    # Encoder
    x1 = self.inc(x)                      # [B, 64, H, W]
    x2 = self.conv1(self.down1(x1))       # [B, 128, H/2, W/2]
    x3 = self.conv2(self.down2(x2))       # [B, 256, H/4, W/4]
    x4 = self.conv3(self.down3(x3))       # [B, 512, H/8, W/8]
    x5 = self.bottleneck(self.down4(x4))  # [B, 1024, H/16, W/16]

    # Decoder
    x = self.up1(x5)                      # [B, 512, H/8, W/8]
    x = torch.cat([x, x4], dim=1)         # Concatenate skip (512+512)
    x = self.conv4(x)

    x = self.up2(x)                       # [B, 256, H/4, W/4]
    x = torch.cat([x, x3], dim=1)
    x = self.conv5(x)

    x = self.up3(x)                       # [B, 128, H/2, W/2]
    x = torch.cat([x, x2], dim=1)
    x = self.conv6(x)

    x = self.up4(x)                       # [B, 64, H, W]
    x = torch.cat([x, x1], dim=1)
    x = self.conv7(x)

    return torch.sigmoid(self.outc(x))